In [1]:
%pip install pypdf sentence-transformers

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   --------------------------------------- 740.6/740.6 kB 29.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   --------------------------------------- 846.4/846.4 kB 36.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 64.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.7 MB ? eta -:--:--
   ----------------- ---------------------- 15.7/36.7 MB 82.7 MB/s eta 0:00:01
   ------------------------------------ --- 33.0/36.7 MB 84.0 MB/s eta 0:00:01
   ---------------------------------------- 36.7/36.7 MB 75.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 55.5 MB/s eta 0:00:00
   ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] T

In [1]:
from agents.db import get_connection
conn = get_connection()

c:\Users\agast\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\snowflake\connector\connection\_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


In [2]:
import glob, os
from pypdf import PdfReader

def chunk_text(text, size=1000, overlap=200):
    """Split text into ~size-character chunks without cutting words in half."""
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        length, end = 0, start
        while end < len(words) and length + len(words[end]) + 1 <= size:
            length += len(words[end]) + 1
            end += 1
        end = max(end, start + 1)
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        # step back about `overlap` characters, measured in whole words
        back, back_len = end, 0
        while back > start + 1 and back_len < overlap:
            back -= 1
            back_len += len(words[back]) + 1
        start = back
    return chunks

docs = []
for path in glob.glob("policy_docs/*.pdf"):
    text = " ".join((page.extract_text() or "") for page in PdfReader(path).pages)
    text = " ".join(text.split())
    for i, ch in enumerate(chunk_text(text)):
        docs.append((os.path.basename(path), i, ch))

print(len(docs), "chunks from", len({d[0] for d in docs}), "PDFs")

63 chunks from 6 PDFs


In [3]:
print("Files in policy_docs:")
for path in glob.glob("policy_docs/*"):
    print("  ", path)

print("\nText found in each PDF:")
for path in glob.glob("policy_docs/*.pdf"):
    text = " ".join((page.extract_text() or "") for page in PdfReader(path).pages)
    print("  ", os.path.basename(path), "→", len(text.strip()), "characters")

Files in policy_docs:
   policy_docs\access_in_motion.pdf
   policy_docs\boston_closure_shuttle_policy.pdf
   policy_docs\mbta_accessibility_overview.pdf
   policy_docs\mbta_elevator_alerts.pdf
   policy_docs\sources.md
   policy_docs\station_downtown_crossing.pdf
   policy_docs\the_ride.pdf

Text found in each PDF:
   access_in_motion.pdf → 17908 characters
   boston_closure_shuttle_policy.pdf → 8274 characters
   mbta_accessibility_overview.pdf → 6737 characters
   mbta_elevator_alerts.pdf → 6548 characters
   station_downtown_crossing.pdf → 1768 characters
   the_ride.pdf → 7875 characters


In [4]:
import json
from agents.retriever import embed

cur = conn.cursor()
cur.execute("TRUNCATE TABLE TRANSITWISE_DB.RAG.POLICY_CHUNKS")

for source, n, ch in docs:
    cur.execute("""
        INSERT INTO TRANSITWISE_DB.RAG.POLICY_CHUNKS
        SELECT %s, %s, %s, PARSE_JSON(%s)::ARRAY::VECTOR(FLOAT, 384)
    """, (source, n, ch, json.dumps(embed(ch))))

print("Loaded", len(docs), "chunks into Snowflake")

c:\Users\agast\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2395.64it/s]


Loaded 63 chunks into Snowflake


In [ ]:
from agents.retriever import search_policies

results = search_policies(conn, "The elevator is out of service at Downtown Crossing. What should I do?")

for r in results:
    print(f"Score: {r['score']}   Source: {r['source']}")
    print(r["chunk"][:400])
    print("-" * 60)

NameError: name 'results' is not defined